# Sprint 5 – Notebook 12 – Train, Validation & Test Data



In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
rng = np.random.default_rng(42)
df = pd.DataFrame({'Age': rng.integers(18, 65, 100), 'Income': rng.integers(20000, 120000, 100)})
df['Purchased'] = ((df['Age'] > 35) & (df['Income'] > 50000)).astype(int)
print(df.head())

   Age  Income  Purchased
0   22  103273          0
1   54   39990          0
2   48  100489          1
3   38   20736          0
4   38   99684          1


## 1. Training Dataset

### Definition
The training dataset is the portion used to learn model parameters. It contains input features and known target values. The model repeatedly learns patterns from these examples. Training performance alone does not prove generalization.

### Business Example
A store trains a model on historical customer purchases. The model learns relationships between age, income, and purchases. These examples form the training dataset. New customers should be evaluated separately.

### AI/ML Use
Fit preprocessing and model parameters using training data. Keep evaluation data separate. A larger training set can improve learning when examples are representative. Avoid training on future information.

### Python Example

In [3]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
print("Training rows:", len(train_df))

Training rows: 80


### Explanation
The split assigns eighty percent of rows to training. The training set is used to fit the model. Test rows remain separate. This prevents the model from seeing its final evaluation examples.

## 2. Validation Dataset

### Definition
A validation dataset helps choose models and hyperparameters. It is separate from training examples. It provides feedback during development. Repeated tuning on validation data can still overfit it.

### Business Example
A retailer compares two prediction models on a validation set. The better model is selected for further testing. Validation results guide decisions. Final test data remains untouched.

### AI/ML Use
Use validation for tuning thresholds, features, and hyperparameters. Do not fit preprocessing on validation data. Use cross-validation when data is limited. Reserve the test set for final evaluation.

### Python Example

In [5]:
train_part, val_part = train_test_split(train_df, test_size=0.25, random_state=42)
print("Train:", len(train_part), "Validation:", len(val_part))

Train: 60 Validation: 20


### Explanation
The first split already reserved test data. The second split takes validation rows from the remaining training portion. This creates separate development partitions. Neither validation nor test should be used to fit the scaler.

## 3. Test Dataset

### Definition
The test dataset estimates how the final model works on unseen examples. It must remain untouched during model selection. It should represent the expected real-world data. Reusing it for tuning makes the estimate optimistic.

### Business Example
A bank tests its completed fraud model on held-out transactions. The team checks how well it detects new fraud cases. The test set was not used for training. Its results are therefore more meaningful.

### AI/ML Use
Evaluate the final selected pipeline once on test data. Use appropriate metrics for the problem. Never rebalance the test distribution for convenience. Investigate performance differences before deployment.

### Python Example

In [7]:
print("Test rows:", len(test_df))
print("Test targets:", test_df["Purchased"].value_counts().to_dict())

Test rows: 20
Test targets: {0: 11, 1: 9}


### Explanation
The test set contains twenty rows. Its target counts show the class distribution. It is reserved for final evaluation. All preprocessing must be learned elsewhere.

## 4. Train-Test Split

### Definition
A train-test split divides data into learning and evaluation portions. Common ratios include 80/20 or 70/30. The correct ratio depends on data size and problem type. A split must avoid leaking related records across partitions.

### Business Example
A business has one hundred customer records. It may train on eighty and test on twenty. The model learns from one group. Performance is checked on the other group.

### AI/ML Use
Use train_test_split for independent tabular examples. Consider stratification for classification. Use group-aware splits for repeated customers. Use chronological splits for forecasting.

### Python Example

In [9]:
X = df[["Age", "Income"]]
y = df["Purchased"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_tr.shape, X_te.shape)

(80, 2) (20, 2)


### Explanation
The feature matrix contains age and income. The target is purchase status. Eighty rows go to training and twenty to testing. The random state makes the split repeatable.

## 5. Train-Validation-Test Split

### Definition
Three-way splitting separates fitting, tuning, and final evaluation. Training learns parameters. Validation helps select settings. Testing estimates final generalization. Each partition has a different role.

### Business Example
An online store trains several candidate models. Validation data helps choose one. The final model is then checked on test data. This reduces biased evaluation.

### AI/ML Use
A common split is 60/20/20 or 70/15/15. Use stratification for imbalanced classification. Avoid overlapping rows. Fit transformations on training only.

### Python Example

In [11]:
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, stratify=y_trainval, random_state=42)
print(len(X_train), len(X_val), len(X_test))

60 20 20


### Explanation
The first split reserves twenty percent for testing. The second reserves a quarter of the remaining eighty percent for validation. Final proportions are 60/20/20. Stratification approximately preserves class proportions.

## 6. Random State

### Definition
Random state controls reproducible random operations. The same seed produces the same split on the same data. It makes debugging and comparisons easier. It does not automatically make a split statistically valid.

### Business Example
Two analysts compare models using the same customer records. A fixed random state gives them the same train-test partition. Their results become easier to compare. They still must check leakage and sampling bias.

### AI/ML Use
Set random_state for reproducible experiments. Use stratification or groups when needed. Try multiple seeds to check stability. Never choose a seed only because it gives the best test score.

### Python Example

In [13]:
a, _ = train_test_split(df.index, test_size=0.2, random_state=7)
b, _ = train_test_split(df.index, test_size=0.2, random_state=7)
print("Same split:", list(a) == list(b))

Same split: True


### Explanation
Both splits use the same seed. They therefore select the same training indices. This supports reproducibility. It does not prevent leakage by itself.

## 7. Stratified Sampling

### Definition
Stratified sampling preserves class proportions across splits. It is useful when some classes are rare. It reduces the chance of losing a minority class in a small partition. It does not correct imbalance by itself.

### Business Example
A bank has five percent fraudulent transactions. Random splitting might place very few fraud cases in the test set. Stratification maintains similar fraud proportions. Evaluation becomes more stable.

### AI/ML Use
Use stratify=y for classification with enough samples per class. It is not generally appropriate for continuous regression targets. Group and time constraints may take priority. Check class counts after splitting.

### Python Example

In [15]:
_, _, y_a, y_b = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Full:", y.value_counts(normalize=True).round(2).to_dict())
print("Test:", y_b.value_counts(normalize=True).round(2).to_dict())

Full: {0: 0.52, 1: 0.48}
Test: {1: 0.5, 0: 0.5}


### Explanation
The code splits using target labels. The resulting test proportions are close to the full dataset. This makes class representation more reliable. It does not create additional minority examples.

## 8. Data Leakage

### Definition
Data leakage occurs when information unavailable at prediction time influences model training. It can produce misleadingly high scores. Leakage may come from targets, future events, or preprocessing. It often causes poor performance after deployment.

### Business Example
A model predicts whether a customer will cancel a subscription. A feature containing the final cancellation reason reveals the outcome. The model appears highly accurate. It cannot use that information before cancellation.

### AI/ML Use
Check every feature for availability at prediction time. Split before learning preprocessing parameters. Avoid target-derived features. Use pipelines to enforce correct training behavior.

### Python Example

In [17]:
leaky = df.copy()
leaky["Target_Copy"] = leaky["Purchased"]
print("Leaky feature matches target:", (leaky["Target_Copy"] == leaky["Purchased"]).all())

Leaky feature matches target: True


### Explanation
The new column is an exact copy of the target. It would give away the correct answer. A model using this column cannot be trusted. Such features must be excluded before training.

## 9. Temporal Splitting

### Definition
Temporal splitting trains on earlier observations and tests on later ones. It follows the order information arrives. It is important for forecasting and evolving systems. Random splitting can accidentally use future patterns.

### Business Example
A shop predicts sales for October. It trains on sales from January through August. September can be used for validation. October is held out for testing.

### AI/ML Use
Sort observations by timestamp before splitting. Use expanding-window validation when appropriate. Avoid features that look ahead. Account for reporting delays and changing patterns.

### Python Example

In [19]:
sales = pd.DataFrame({"Date": pd.date_range("2026-01-01", periods=10), "Sales": range(10, 20)})
past, future = sales.iloc[:7], sales.iloc[7:]
print("Last training date:", past.Date.max().date())
print("First test date:", future.Date.min().date())

Last training date: 2026-01-07
First test date: 2026-01-08


### Explanation
The first seven dates are training observations. The final three are held out as later data. No future row appears in training. This better matches forecasting than random splitting.

## 10. Why Preprocessing Must Be Fitted Only on Training Data

### Definition
Preprocessing learns values such as means, medians, and scaling ranges. Learning these from test data leaks information. Even unsupervised transformations can cause contamination. The same fitted transformation should be applied to validation and test data.

### Business Example
A salary model scales income using a mean and standard deviation. If it calculates these from all customers, test customers influence training. The reported evaluation may be optimistic. The correct mean comes from training customers only.

### AI/ML Use
Split first, then fit the imputer, encoder, or scaler on training data. Use transform on validation and test data. A Pipeline makes this process safer. Cross-validation should refit preprocessing inside each training fold.

### Python Example

In [21]:
scaler = StandardScaler().fit(X_train)
train_scaled = scaler.transform(X_train)
val_scaled = scaler.transform(X_val)
test_scaled = scaler.transform(X_test)
print("Train mean after scaling:", np.round(train_scaled.mean(axis=0), 3))

Train mean after scaling: [-0. -0.]


### Explanation
The scaler learns its mean and standard deviation from training rows only. Validation and test rows use those same learned values. They do not influence the fitted parameters. This prevents preprocessing leakage.